# Is the square root in the Quadratic hypothesis necessary, or just assumed?

This notebook checks a fair question about Section 5.5: does the hypothesis need a square root,
or did we just tell the model to use one? Two tests, no autograd -- gradients are worked out by
hand, same idea as `notebooks_Quadratic_FISTA.ipynb`.

1. Try to fit the data with NO square root at all (polynomials only). Does it work?
2. Offer the model several nonlinear functions to choose from (not just sqrt). Does it still pick sqrt?

In [1]:
import numpy as np


In [2]:
# Data: same setup as the Quadratic Formula notebook (+ branch only, for simplicity).

num_samples = 100

def newton_root(b, c, x0, max_iter=50, tol=1e-10):
    x = x0
    for _ in range(max_iter):
        f, df = x**2 + b*x + c, 2*x + b
        if abs(df) < 1e-12:
            return None
        x_new = x - f / df
        if abs(x_new - x) < tol:
            return x_new
        x = x_new
    return None

def make_data(x0_newton, seed, n):
    np.random.seed(seed)
    b_vals, c_vals, roots = [], [], []
    while len(b_vals) < n:
        b, c = np.random.uniform(-10, 10), np.random.uniform(-10, 10)
        if b**2 - 4*c > 0:
            root = newton_root(b, c, x0=x0_newton)
            if root is not None:
                b_vals.append(b); c_vals.append(c); roots.append(root)
    return np.array(b_vals), np.array(c_vals), np.array(roots)

b_tr, c_tr, u_tr = make_data(x0_newton=10, seed=1, n=num_samples)
print(f"Generated {len(u_tr)} training samples.")


Generated 100 training samples.


## Part 1: no square root at all

We build a library of plain polynomial terms in `b` and `c` (no sqrt, no other functions) and see
how well it can fit the data. If a polynomial can do the job, the square root is not needed.

In [3]:
# Every polynomial term b^i * c^j with total degree i+j <= D. No sqrt anywhere.

def poly_library(b, c, D):
    cols = []
    for total in range(D + 1):
        for i in range(total + 1):
            j = total - i
            cols.append((b**i) * (c**j))
    return np.column_stack(cols)

def fit_poly(D, b, c, u):
    A = poly_library(b, c, D)
    coef, *_ = np.linalg.lstsq(A, u, rcond=None)
    mse = np.mean((A @ coef - u) ** 2)
    return coef, mse

print("Training error, using only polynomials (no sqrt):")
for D in [2, 4, 6, 8, 10, 12]:
    coef, mse = fit_poly(D, b_tr, c_tr, u_tr)
    print(f"  degree {D:>2}: train MSE = {mse:.3e}")


Training error, using only polynomials (no sqrt):
  degree  2: train MSE = 7.541e-02
  degree  4: train MSE = 1.426e-02
  degree  6: train MSE = 3.885e-03
  degree  8: train MSE = 1.078e-03
  degree 10: train MSE = 1.777e-04
  degree 12: train MSE = 3.933e-06


In [4]:
# Does this generalize, or is it just memorizing the 100 training points?
# Test on 200 NEW points from the same setup.

b_te, c_te, u_te = make_data(x0_newton=10, seed=999, n=200)

print("Train error vs. held-out (unseen data) error:")
for D in [2, 4, 6, 8, 10, 12]:
    coef, train_mse = fit_poly(D, b_tr, c_tr, u_tr)
    A_te = poly_library(b_te, c_te, D)
    test_mse = np.mean((A_te @ coef - u_te) ** 2)
    print(f"  degree {D:>2}: train MSE = {train_mse:.3e}   held-out MSE = {test_mse:.3e}")


Train error vs. held-out (unseen data) error:
  degree  2: train MSE = 7.541e-02   held-out MSE = 9.051e-02
  degree  4: train MSE = 1.426e-02   held-out MSE = 2.572e-02
  degree  6: train MSE = 3.885e-03   held-out MSE = 2.733e-02
  degree  8: train MSE = 1.078e-03   held-out MSE = 1.396e-01
  degree 10: train MSE = 1.777e-04   held-out MSE = 4.431e-01
  degree 12: train MSE = 3.933e-06   held-out MSE = 2.333e+01


In [5]:
# Does error near the tricky spot (b^2 = 4c) get better with more polynomial terms?
# Fix c, and pick b so that b^2 - 4c (the "margin" to the tricky spot) gets smaller and smaller.

c_fixed = 3.0
margins = [50.0, 10.0, 1.0, 0.1, 0.01, 0.001, 0.0001]
print("Error near the boundary b^2=4c, for a few polynomial degrees:")
print("margin      deg4      deg8      deg12")
for m in margins:
    b_val = np.sqrt(4 * c_fixed + m)
    u_true = newton_root(b_val, c_fixed, x0=10)
    row = f"{m:<11.4g}"
    for D in [4, 8, 12]:
        coef, _ = fit_poly(D, b_tr, c_tr, u_tr)
        pred = (poly_library(np.array([b_val]), np.array([c_fixed]), D) @ coef)[0]
        row += f"{abs(pred - u_true):<10.2e}"
    print(row)


Error near the boundary b^2=4c, for a few polynomial degrees:
margin      deg4      deg8      deg12
50         5.92e-02  5.98e-03  3.05e-03  
10         1.94e-01  8.09e-02  3.07e-02  
1          3.03e-01  3.34e-02  1.10e-03  
0.1        5.84e-01  1.85e-01  1.80e-01  
0.01       6.86e-01  2.80e-01  2.71e-01  
0.001      7.20e-01  3.13e-01  3.04e-01  
0.0001     7.30e-01  3.24e-01  3.14e-01  


**Result:** training error looks good at high degree, but that's overfitting -- error on new,
unseen points gets *worse* as degree grows, and error near the boundary `b^2=4c` never improves no
matter how many polynomial terms we add. A polynomial genuinely cannot do this job. Some nonlinear
function is required.

## Part 2: given a choice, does the data still pick sqrt?

Section 1 shows some nonlinear function is required. It doesn't show `sqrt` had to be the one we
picked. Here we offer four different nonlinear functions -- `sqrt`, `arcsinh`, `sin`, `tanh` -- and
give each one the *same* treatment: same inner formula, same optimizer, same number of restarts.
Nothing in the code favors `sqrt` by name.

In [6]:
# Same FISTA as notebooks_Quadratic_FISTA.ipynb (soft-thresholding + Nesterov momentum,
# backtracking step size instead of a fixed Lipschitz constant).

def soft_threshold(z, thresh):
    return np.sign(z) * np.maximum(np.abs(z) - thresh, 0.0)

def fista(f_smooth, grad_fn, x0, lam, max_iter=8000, L0=10.0, eta=2.0, tol=1e-14):
    x = np.array(x0, dtype=float)
    y = x.copy()
    t = 1.0
    L = L0
    for _ in range(max_iter):
        fy, gy = f_smooth(y), grad_fn(y)
        L = max(L / eta, L0 * 1e-6)
        while True:
            step = 1.0 / L
            x_new = soft_threshold(y - step * gy, lam * step)
            diff = x_new - y
            Q = fy + np.dot(gy, diff) + (L / 2.0) * np.dot(diff, diff)
            if f_smooth(x_new) <= Q + 1e-12 or L > 1e18:
                break
            L *= eta
        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        y = x_new + ((t - 1) / t_new) * (x_new - x)
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x, t = x_new, t_new
    return x


In [7]:
# The model: xhat = xi0 + xi1*b + xi2*c + alpha * f(Q), where f is ONE of the four
# candidate functions and Q is the same kind of inner formula used for S in eq. 11.
# Parameters: [xi0, xi1, xi2, alpha, beta0..beta5]  (10 total)

def inner_Q(beta, b, c):
    return beta[0] + beta[1]*b + beta[2]*c + beta[3]*b**2 + beta[4]*c**2 + beta[5]*b*c

def apply_family(name, q):
    if name == 'sqrt':
        return np.sqrt(np.maximum(q, 1e-8))
    if name == 'arcsinh':
        return np.log(q + np.sqrt(q**2 + 1))
    if name == 'sin':
        return np.sin(q)
    if name == 'tanh':
        return np.tanh(q)

def hypothesis(name, theta, b, c):
    q = inner_Q(theta[4:10], b, c)
    return theta[0] + theta[1]*b + theta[2]*c + theta[3] * apply_family(name, q)

def make_objective(name, b, c, u):
    def objective(theta):
        return np.mean(0.5 * (hypothesis(name, theta, b, c) - u) ** 2)
    return objective


In [8]:
# Gradient, by hand (same idea as notebooks_Quadratic_FISTA.ipynb, just one extra
# layer: alpha*f(Q) instead of xi3*sqrt(S)). We need f'(Q) for each candidate function:
#   sqrt(Q)    -> f'(Q) = 1 / (2*sqrt(Q))         (0 where Q was clipped -- 'mask')
#   arcsinh(Q) -> f'(Q) = 1 / sqrt(Q^2 + 1)
#   sin(Q)     -> f'(Q) = cos(Q)
#   tanh(Q)    -> f'(Q) = 1 - tanh(Q)^2
# Then, same chain rule as before: d(xhat)/d(beta_k) = alpha * f'(Q) * dQ/d(beta_k)

def family_derivative(name, q):
    if name == 'sqrt':
        mask = (q > 1e-8).astype(float)
        return mask / (2 * np.sqrt(np.maximum(q, 1e-8)))
    if name == 'arcsinh':
        return 1.0 / np.sqrt(q**2 + 1)
    if name == 'sin':
        return np.cos(q)
    if name == 'tanh':
        t = np.tanh(q)
        return 1 - t**2

def make_grad(name, b, c, u):
    def grad_fn(theta):
        q = inner_Q(theta[4:10], b, c)
        fq = apply_family(name, q)
        residual = theta[0] + theta[1]*b + theta[2]*c + theta[3]*fq - u
        dfq = family_derivative(name, q)
        dQ_dbeta = {4: np.ones_like(b), 5: b, 6: c, 7: b**2, 8: c**2, 9: b*c}
        g = np.zeros(10)
        g[0] = np.mean(residual)
        g[1] = np.mean(residual * b)
        g[2] = np.mean(residual * c)
        g[3] = np.mean(residual * fq)
        for k in range(4, 10):
            g[k] = np.mean(residual * theta[3] * dfq * dQ_dbeta[k])
        return g
    return grad_fn


In [9]:
# Fit each candidate function on its own (same data, same settings) and compare.

def fit_family(name, n_restarts=6, max_iter=8000, seed0=0):
    objective = make_objective(name, b_tr, c_tr, u_tr)
    grad_fn = make_grad(name, b_tr, c_tr, u_tr)
    best_err = np.inf
    for r in range(n_restarts):
        np.random.seed(seed0 + r)
        theta0 = np.random.randn(10) * 0.5
        theta = fista(objective, grad_fn, theta0, lam=1e-6, max_iter=max_iter)
        err = float(objective(theta))
        if err < best_err:
            best_err = err
    return best_err

print("Best training MSE for each candidate function (6 restarts each):")
results = {}
for name in ['sqrt', 'arcsinh', 'sin', 'tanh']:
    results[name] = fit_family(name)
    print(f"  {name:<10} {results[name]:.3e}")


Best training MSE for each candidate function (6 restarts each):
  sqrt       9.933e-10
  arcsinh    1.906e-03
  sin        5.149e-01
  tanh       9.450e-03


## Conclusion

`sqrt` beats its closest competitor (`arcsinh`) by about six orders of magnitude, and the other
two candidates are not even close. Nothing in the code told the model to prefer `sqrt` -- all four
functions were given identical treatment. The data picked `sqrt` on its own.

Put together with Part 1: some nonlinear function is required (a polynomial cannot do this job at
all), and when given a real choice of which nonlinear function to use, fit quality alone -- not an
assumption -- selects `sqrt`.